# Feature Engineering con Análisis de Sentimiento
### Caso aplicado: Opinión pública en redes sobre candidatos al Congreso de Colombia 2026

---

**Curso:** Analítica de Datos y Machine Learning — Ciencias Políticas, Universidad de Antioquia  
**Objetivo:** Aprender a usar **dos modelos complementarios** de la librería `pysentimiento` para crear nuevas variables (features) a partir de texto no estructurado en español.

---

### ¿Qué vamos a hacer?

Imaginemos que estamos monitoreando la conversación en redes sociales (tipo Twitter/X) sobre cinco candidatos ficticios al Congreso de Colombia. Tenemos un dataset de tweets crudos y queremos extraer información analítica de ellos.

Vamos a usar **2 modelos de `pysentimiento`** de forma complementaria:

| Modelo | Tarea | ¿Qué extrae? | Rol en el pipeline |
|---|---|---|---|
| Sentimiento | `sentiment` | POS / NEG / NEU + probabilidades | Clasificación base de todos los tweets |
| Emociones | `emotion` | joy, sadness, anger, surprise, disgust, fear, others | Enriquece solo los tweets negativos |

La idea clave es la misma del notebook anterior: **el Modelo 1 hace el trabajo general** y **el Modelo 2 profundiza en los casos más interesantes**. Si un tweet es negativo, ¿es por tristeza, rabia o asco? Eso tiene implicaciones políticas muy distintas.

### ¿Por qué es esto Feature Engineering?

Partimos de una sola columna de texto libre (`tweet`) y vamos a crear **múltiples features numéricas y categóricas**:

| Feature creada | Tipo | Ejemplo |
|---|---|---|
| `sentimiento` | Categórica | POS, NEG, NEU |
| `prob_positivo` | Numérica [0,1] | 0.85 |
| `prob_negativo` | Numérica [0,1] | 0.10 |
| `prob_neutro` | Numérica [0,1] | 0.05 |
| `emocion` | Categórica | anger, sadness, joy... |
| `sentimiento_final` | Categórica | Combinación de ambos modelos |

Estas features luego pueden usarse para dashboards de monitoreo electoral, modelos predictivos de intención de voto, o análisis de narrativas en campañas.

### Sobre `pysentimiento`

[`pysentimiento`](https://github.com/pysentimiento/pysentimiento) es una librería open-source desarrollada por investigadores argentinos (Pérez et al., 2023), construida sobre Hugging Face Transformers. Su gran ventaja es que ofrece modelos **pre-entrenados específicamente en español** y optimizados para texto de redes sociales (tweets, posts), lo cual es ideal para nuestro caso.

---
### Flujo general del notebook

```
┌─────────────────────────────────────────────────────────────────────┐
│  DATOS CRUDOS: candidato + tweet (texto libre)                      │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  PASO 1: MODELO 1 (Sentimiento) → clasifica TODOS los tweets       │
│          Genera: sentimiento + prob_positivo/negativo/neutro        │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                  ┌────────┴────────┐
                  │                 │
          POS o NEU              NEG
        (se quedan así)    (queremos saber más)
                  │                 │
                  │                 ▼
                  │  ┌────────────────────────────────────────────────┐
                  │  │  PASO 2: MODELO 2 (Emociones) → solo los NEG  │
                  │  │          Genera: emocion (anger/sadness/...)   │
                  │  └──────────────────┬─────────────────────────────┘
                  │                     │
                  └────────┬────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  PASO 3: RECONCILIACIÓN → sentimiento_final                         │
│          Combina sentimiento + emoción en una etiqueta enriquecida  │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  PASO 4: ANÁLISIS EXPLORATORIO por candidato                        │
│          ¿Quién genera más rabia? ¿Quién más alegría?               │
└─────────────────────────────────────────────────────────────────────┘
```

---
## Paso 1: Instalación de dependencias

`pysentimiento` se instala directamente desde PyPI. Internamente descarga modelos de Hugging Face, por lo que la primera ejecución puede tardar un poco.

In [ ]:
!pip install -q pysentimiento pandas plotly tqdm

---
## Paso 2: Crear el dataset de tweets simulados

En un proyecto real, estos datos vendrían de la API de Twitter/X, de scraping, o de una base de datos existente. Para este ejercicio pedagógico, vamos a crear un dataset ficticio con **5 candidatos** y **tweets inventados** que representen distintos tipos de opinión pública.

Los tweets están diseñados para cubrir casos variados: opiniones claramente positivas, claramente negativas, neutras, irónicas, con jerga colombiana, y casos ambiguos. Esto nos permitirá ver cómo se comportan los modelos en distintos escenarios.

In [ ]:
import pandas as pd

# Dataset de tweets simulados sobre candidatos ficticios
tweets_data = [
    # ===== CANDIDATO A — perfil: genera polarización =====
    {"candidato": "Candidato A", "tweet": "El candidato A es el mejor que ha tenido este país, tiene todas las propuestas claras"},
    {"candidato": "Candidato A", "tweet": "Qué propuestas tan buenas las del candidato A, por fin alguien habla de educación"},
    {"candidato": "Candidato A", "tweet": "Me encanta lo que propone el candidato A para el campo colombiano"},
    {"candidato": "Candidato A", "tweet": "El candidato A es un mentiroso, no le crean nada de lo que dice"},
    {"candidato": "Candidato A", "tweet": "Otro corrupto más, el candidato A tiene investigaciones pendientes"},
    {"candidato": "Candidato A", "tweet": "Qué rabia que el candidato A siga engañando a la gente con promesas vacías"},
    {"candidato": "Candidato A", "tweet": "El candidato A habló hoy en el debate de Caracol sobre reforma agraria"},
    {"candidato": "Candidato A", "tweet": "Vi la entrevista del candidato A, no dijo nada nuevo la verdad"},
    {"candidato": "Candidato A", "tweet": "Jajaja el candidato A se quedó sin argumentos en el debate, qué oso"},
    {"candidato": "Candidato A", "tweet": "El candidato A tiene 45 años y es abogado de la Universidad Nacional"},

    # ===== CANDIDATO B — perfil: genera entusiasmo =====
    {"candidato": "Candidato B", "tweet": "La candidata B es una berraquera, ojalá gane"},
    {"candidato": "Candidato B", "tweet": "Increíble el discurso de la candidata B ayer, me hizo llorar de emoción"},
    {"candidato": "Candidato B", "tweet": "La candidata B es la esperanza que necesitamos en el Congreso"},
    {"candidato": "Candidato B", "tweet": "La candidata B no tiene experiencia suficiente para el cargo"},
    {"candidato": "Candidato B", "tweet": "Me preocupa que la candidata B no tenga equipo técnico serio"},
    {"candidato": "Candidato B", "tweet": "La candidata B va a participar en el foro de Medellín este viernes"},
    {"candidato": "Candidato B", "tweet": "Qué tristeza que la candidata B no haya hablado del desempleo juvenil"},
    {"candidato": "Candidato B", "tweet": "La candidata B recibió el apoyo de varios sindicatos hoy"},
    {"candidato": "Candidato B", "tweet": "No sé qué pensar de la candidata B, dice cosas bonitas pero no convence"},
    {"candidato": "Candidato B", "tweet": "Excelente propuesta de la candidata B sobre salud mental en colegios"},

    # ===== CANDIDATO C — perfil: genera rechazo fuerte =====
    {"candidato": "Candidato C", "tweet": "El candidato C da asco, es de lo peor que hay en la política colombiana"},
    {"candidato": "Candidato C", "tweet": "Qué miedo que el candidato C llegue al Congreso, sería un desastre"},
    {"candidato": "Candidato C", "tweet": "Me indigna ver al candidato C haciendo campaña después de todo lo que hizo"},
    {"candidato": "Candidato C", "tweet": "El candidato C es un peligro para la democracia, no lo apoyen"},
    {"candidato": "Candidato C", "tweet": "El candidato C se inscribió por el partido Verde Esperanza"},
    {"candidato": "Candidato C", "tweet": "Sorprendente que el candidato C tenga tantos votos a pesar de todo"},
    {"candidato": "Candidato C", "tweet": "El candidato C propone privatizar la salud, qué tristeza por los pobres"},
    {"candidato": "Candidato C", "tweet": "Uy no, el candidato C otra vez hablando pendejadas en televisión"},
    {"candidato": "Candidato C", "tweet": "Alguien me explica por qué el candidato C sigue en las encuestas?"},
    {"candidato": "Candidato C", "tweet": "El candidato C al menos tiene claro su plan de seguridad, eso hay que reconocerlo"},

    # ===== CANDIDATO D — perfil: poco conocido, neutro =====
    {"candidato": "Candidato D", "tweet": "¿Alguien sabe quién es el candidato D? Primera vez que lo escucho"},
    {"candidato": "Candidato D", "tweet": "El candidato D presentó su plan de gobierno ante la MOE"},
    {"candidato": "Candidato D", "tweet": "No conozco al candidato D pero dicen que tiene buenas ideas"},
    {"candidato": "Candidato D", "tweet": "El candidato D es otro más del montón, nada diferente"},
    {"candidato": "Candidato D", "tweet": "Me gusta que el candidato D hable de tecnología y datos abiertos"},
    {"candidato": "Candidato D", "tweet": "El candidato D tiene experiencia en el concejo de Bucaramanga"},
    {"candidato": "Candidato D", "tweet": "El candidato D dijo que va a impulsar la economía naranja, interesante"},
    {"candidato": "Candidato D", "tweet": "Aburrido el debate del candidato D, se la pasó leyendo"},
    {"candidato": "Candidato D", "tweet": "El candidato D es joven y eso me da esperanza, necesitamos gente nueva"},
    {"candidato": "Candidato D", "tweet": "La verdad el candidato D no me genera nada, ni bien ni mal"},

    # ===== CANDIDATO E — perfil: controversial, ironía =====
    {"candidato": "Candidato E", "tweet": "Claro que sí, la candidata E va a acabar con la corrupción, como no jajaja"},
    {"candidato": "Candidato E", "tweet": "La candidata E es la única que habla claro sobre el conflicto armado"},
    {"candidato": "Candidato E", "tweet": "Muy valiente la candidata E denunciando la minería ilegal en Chocó"},
    {"candidato": "Candidato E", "tweet": "La candidata E es puro populismo, dice lo que la gente quiere oír"},
    {"candidato": "Candidato E", "tweet": "Me da miedo que la candidata E termine como otros líderes sociales"},
    {"candidato": "Candidato E", "tweet": "La candidata E fue a Tumaco y la gente la recibió con alegría"},
    {"candidato": "Candidato E", "tweet": "Sí claro, la candidata E va a cambiar el país desde el Congreso, eso nunca pasa"},
    {"candidato": "Candidato E", "tweet": "Respeto a la candidata E aunque no comparto sus ideas económicas"},
    {"candidato": "Candidato E", "tweet": "La candidata E lloró en la entrevista hablando de las víctimas, muy conmovedor"},
    {"candidato": "Candidato E", "tweet": "La candidata E habla mucho pero al final todos los políticos son iguales"},
]

df = pd.DataFrame(tweets_data)

print(f"Total de tweets: {len(df)}")
print(f"Candidatos: {df['candidato'].nunique()}")
print(f"Tweets por candidato: {df['candidato'].value_counts().to_dict()}")
print("=" * 60)
df.head(10)

---
## Paso 3: Modelo 1 — Análisis de Sentimiento

📦 **Tarea:** `sentiment`  
🤖 **Modelo interno:** `pysentimiento/robertuito-sentiment-analysis` (basado en RoBERTuito, un modelo pre-entrenado con +500 millones de tweets en español)  
📊 **Output:** POS (positivo), NEG (negativo), NEU (neutro) + probabilidades para cada clase  

### ¿Qué es RoBERTuito?

`pysentimiento` usa internamente un modelo llamado **RoBERTuito**, que es un BERT pre-entrenado específicamente con tweets en español. Esto es crucial: un modelo entrenado con textos formales (Wikipedia, libros) tiende a fallar con el lenguaje coloquial de redes sociales. RoBERTuito entiende abreviaciones, jerga, emojis y el tono informal de Twitter.

### ¿Cómo funciona `create_analyzer`?

La función `create_analyzer()` abstrae toda la complejidad: descarga el modelo desde Hugging Face, lo carga en memoria, y expone un método `.predict()` que recibe texto y devuelve la clasificación con probabilidades.

In [ ]:
from pysentimiento import create_analyzer

# Cargamos el analizador de sentimiento en español
# La primera vez descarga el modelo (~500 MB), luego queda en caché
sentiment_analyzer = create_analyzer(task="sentiment", lang="es")

print("Modelo de sentimiento cargado exitosamente")

### Prueba rápida del Modelo 1

Antes de aplicar a todo el dataset, probemos con ejemplos controlados para entender la estructura del output.

In [ ]:
# Probemos con frases que conocemos
pruebas = [
    "Excelente candidato, tiene mi voto",
    "Este político es un desastre total",
    "El candidato habló hoy en el debate de las 8pm",
    "Jajaja claro que sí, este nos va a salvar, como no",
]

for texto in pruebas:
    resultado = sentiment_analyzer.predict(texto)
    print(f"Texto: {texto}")
    print(f"  → Sentimiento: {resultado.output}")
    print(f"  → Probabilidades: POS={resultado.probas['POS']:.3f}  NEG={resultado.probas['NEG']:.3f}  NEU={resultado.probas['NEU']:.3f}")
    print()

**Observemos varias cosas:**
- Cada predicción tiene una etiqueta ganadora (`.output`) y un diccionario de probabilidades (`.probas`).
- Las probabilidades suman ~1.0 y nos permiten medir la "seguridad" del modelo.
- Los textos irónicos son un desafío: el modelo puede no captar el sarcasmo.
- Los textos informativos tienden a clasificarse como neutros.

### Aplicar Modelo 1 a todos los tweets

In [ ]:
from tqdm.auto import tqdm

# Procesamos todos los tweets
resultados_sentimiento = []
for tweet in tqdm(df['tweet'].tolist(), desc="Modelo 1 (Sentimiento)"):
    res = sentiment_analyzer.predict(tweet)
    resultados_sentimiento.append(res)

# Crear las nuevas features
df['sentimiento'] = [r.output for r in resultados_sentimiento]
df['prob_positivo'] = [round(r.probas['POS'], 4) for r in resultados_sentimiento]
df['prob_negativo'] = [round(r.probas['NEG'], 4) for r in resultados_sentimiento]
df['prob_neutro'] = [round(r.probas['NEU'], 4) for r in resultados_sentimiento]

# Feature derivada: "confianza" = probabilidad máxima
df['confianza_sentimiento'] = df[['prob_positivo', 'prob_negativo', 'prob_neutro']].max(axis=1)

print("\n✅ Modelo 1 completado")
print(f"\nDistribución de sentimiento:")
print(df['sentimiento'].value_counts())
print(f"\nConfianza promedio: {df['confianza_sentimiento'].mean():.4f}")

In [ ]:
df.head(10)

### Inspección de resultados del Modelo 1

Revisemos los casos más interesantes: los tweets donde el modelo está menos seguro, y cómo se distribuye el sentimiento por candidato.

In [ ]:
# Casos con menor confianza (el modelo "duda")
print("🔍 Top 10 tweets con menor confianza del Modelo 1:")
print("=" * 80)
cols_inspeccion = ['candidato', 'tweet', 'sentimiento', 'confianza_sentimiento']
print(df.nsmallest(10, 'confianza_sentimiento')[cols_inspeccion].to_string(index=False))

In [ ]:
# Sentimiento por candidato (tabla cruzada)
tabla_sentimiento = pd.crosstab(df['candidato'], df['sentimiento'], normalize='index')
tabla_sentimiento = tabla_sentimiento.round(2)

print("📊 Proporción de sentimiento por candidato:")
print("=" * 50)
print(tabla_sentimiento.to_string())

---
## Paso 4: Modelo 2 — Análisis de Emociones (solo tweets negativos)

📦 **Tarea:** `emotion`  
🤖 **Modelo interno:** `pysentimiento/robertuito-emotion-analysis`  
📊 **Output:** Las 6 emociones básicas de Ekman + "others"

| Emoción | Código | Descripción política |
|---|---|---|
| `joy` | Alegría | Entusiasmo por propuestas o resultados |
| `sadness` | Tristeza | Decepción, desilusión con el candidato |
| `anger` | Rabia | Indignación, rechazo activo |
| `fear` | Miedo | Temor por consecuencias de su elección |
| `disgust` | Asco | Desprecio moral por el candidato |
| `surprise` | Sorpresa | Asombro (puede ser positivo o negativo) |
| `others` | Otro | No encaja en ninguna emoción clara |

### ¿Por qué solo los tweets negativos?

Esta es la estrategia de **modelos complementarios**: no necesitamos saber la emoción de un tweet positivo (ya sabemos que es positivo). Pero un tweet negativo puede ser negativo por **rabia** (ej: "qué indignación") o por **tristeza** (ej: "qué decepción") o por **miedo** (ej: "me da terror"). Esas distinciones son políticamente relevantes:

- **Rabia** → riesgo de polarización y movilización reactiva
- **Tristeza** → riesgo de desafección y abstención electoral  
- **Miedo** → efectividad de narrativas alarmistas
- **Asco** → rechazo moral profundo, difícil de revertir

In [ ]:
# Cargamos el analizador de emociones en español
emotion_analyzer = create_analyzer(task="emotion", lang="es")

print("Modelo de emociones cargado exitosamente")

### Prueba rápida del Modelo 2

In [ ]:
# Probemos con tweets que sabemos son negativos pero con emociones diferentes
pruebas_negativas = [
    "Qué rabia que sigan robándose la plata del pueblo",          # anger
    "Me da mucha tristeza ver cómo se destruye la salud pública", # sadness
    "Me da miedo que este candidato llegue al poder",             # fear
    "Ese candidato da asco, pura corrupción",                     # disgust
]

for texto in pruebas_negativas:
    res = emotion_analyzer.predict(texto)
    # Ordenar probabilidades de mayor a menor para ver el ranking
    probas_ordenadas = sorted(res.probas.items(), key=lambda x: x[1], reverse=True)
    top3 = probas_ordenadas[:3]
    print(f"Texto: {texto}")
    print(f"  → Emoción dominante: {res.output}")
    print(f"  → Top 3: {', '.join([f'{k}={v:.3f}' for k,v in top3])}")
    print()

### Aplicar Modelo 2 solo a los tweets negativos

Igual que en el notebook anterior, no procesamos todo el dataset con el segundo modelo — solo los casos donde el Modelo 1 detectó sentimiento negativo.

In [ ]:
# 1. Identificar tweets negativos
mask_negativos = df['sentimiento'] == 'NEG'
df_negativos = df[mask_negativos].copy()

print(f"Tweets negativos a procesar con Modelo 2: {len(df_negativos)}")
print(f"Esto es el {len(df_negativos)/len(df)*100:.1f}% del total")

In [ ]:
# 2. Inicializar columnas de emociones como NaN para todos
df['emocion'] = pd.NA
df['prob_emocion_dominante'] = pd.NA

# 3. Ejecutar Modelo 2 solo sobre los negativos
if not df_negativos.empty:
    resultados_emociones = []
    for tweet in tqdm(df_negativos['tweet'].tolist(), desc="Modelo 2 (Emociones)"):
        res = emotion_analyzer.predict(tweet)
        resultados_emociones.append(res)

    # Asignar resultados solo a las filas negativas
    df.loc[mask_negativos, 'emocion'] = [r.output for r in resultados_emociones]
    df.loc[mask_negativos, 'prob_emocion_dominante'] = [
        round(max(r.probas.values()), 4) for r in resultados_emociones
    ]

print("\n✅ Modelo 2 completado")
print(f"\nDistribución de emociones en tweets negativos:")
print(df.loc[mask_negativos, 'emocion'].value_counts())

---
## Paso 5: Reconciliación — Crear la feature enriquecida `sentimiento_final`

Ahora combinamos los resultados de ambos modelos en una sola feature que capture más matices que un simple POS/NEG/NEU.

La regla de decisión:
- Si el tweet es **POS** → `sentimiento_final = "positivo"`
- Si el tweet es **NEU** → `sentimiento_final = "neutro"`
- Si el tweet es **NEG** → `sentimiento_final = "neg_" + emoción dominante` (ej: `"neg_anger"`, `"neg_sadness"`)

Esto nos da una variable con **mayor granularidad** para los casos negativos, que son los más relevantes en monitoreo de crisis y campañas.

In [ ]:
# Crear el sentimiento enriquecido
def crear_sentimiento_final(row):
    if row['sentimiento'] == 'POS':
        return 'positivo'
    elif row['sentimiento'] == 'NEU':
        return 'neutro'
    elif row['sentimiento'] == 'NEG' and pd.notna(row['emocion']):
        return f"neg_{row['emocion']}"
    else:
        return 'negativo_sin_emocion'

df['sentimiento_final'] = df.apply(crear_sentimiento_final, axis=1)

# También creamos una columna de "fuente" para trazabilidad
df['fuente_prediccion'] = df['sentimiento'].apply(
    lambda x: 'Modelo 1 + Modelo 2' if x == 'NEG' else 'Solo Modelo 1'
)

print("✅ Feature 'sentimiento_final' creada exitosamente")
print("=" * 50)
print(f"\nDistribución del sentimiento enriquecido:")
print(df['sentimiento_final'].value_counts())
print(f"\nFuente de cada predicción:")
print(df['fuente_prediccion'].value_counts())

In [ ]:
# Vista del dataset con todas las features
columnas_resumen = ['candidato', 'tweet', 'sentimiento', 'confianza_sentimiento',
                    'emocion', 'sentimiento_final']
df[columnas_resumen].head(15)

### Inspección: ¿Qué emociones dominan lo negativo?

Veamos los tweets negativos con su emoción asignada para validar que tiene sentido.

In [ ]:
# Tweets negativos con sus emociones
cols_neg = ['candidato', 'tweet', 'emocion', 'prob_emocion_dominante']
df_neg_detalle = df[df['sentimiento'] == 'NEG'][cols_neg].copy()

print(f"📋 Detalle de los {len(df_neg_detalle)} tweets negativos y su emoción:")
print("=" * 80)
print(df_neg_detalle.to_string(index=False))

---
## Paso 6: Análisis exploratorio con las nuevas features

Ahora viene lo que realmente le importa a un politólogo: ¿qué nos dicen estos datos sobre la opinión pública hacia cada candidato? Las features que creamos nos permiten ir más allá de un simple conteo.

In [ ]:
import plotly.express as px

# 1. Distribución general de sentimiento
fig = px.pie(
    df, names='sentimiento',
    title='Distribución general de sentimiento en tweets sobre candidatos',
    color='sentimiento',
    color_discrete_map={'POS': '#00CC96', 'NEG': '#EF553B', 'NEU': '#636EFA'},
    hole=0.4
)
fig.update_traces(textinfo='label+percent+value')
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
# 2. Sentimiento por candidato (barras apiladas)
sent_por_candidato = pd.crosstab(df['candidato'], df['sentimiento'])

fig = px.bar(
    sent_por_candidato.reset_index(),
    x='candidato',
    y=['POS', 'NEU', 'NEG'],
    title='Sentimiento por candidato',
    labels={'value': 'Número de tweets', 'candidato': ''},
    color_discrete_map={'POS': '#00CC96', 'NEG': '#EF553B', 'NEU': '#636EFA'},
    barmode='stack'
)
fig.update_layout(template='plotly_white', legend_title='Sentimiento')
fig.show()

In [ ]:
# 3. Emociones dentro de lo negativo, por candidato
df_emociones = df[df['sentimiento'] == 'NEG'].copy()

if not df_emociones.empty:
    emo_por_candidato = pd.crosstab(df_emociones['candidato'], df_emociones['emocion'])

    fig = px.bar(
        emo_por_candidato.reset_index(),
        x='candidato',
        y=emo_por_candidato.columns.tolist(),
        title='Emociones en tweets negativos por candidato<br><sup>¿Qué tipo de negatividad genera cada candidato?</sup>',
        labels={'value': 'Número de tweets', 'candidato': ''},
        color_discrete_sequence=px.colors.qualitative.Set2,
        barmode='stack'
    )
    fig.update_layout(template='plotly_white', legend_title='Emoción', height=500)
    fig.show()
else:
    print("No hay tweets negativos para analizar.")

In [ ]:
# 4. Sentimiento enriquecido (el producto final de nuestro Feature Engineering)
fig = px.histogram(
    df, x='sentimiento_final', color='candidato',
    title='Sentimiento enriquecido por candidato<br><sup>Feature final que combina ambos modelos</sup>',
    labels={'sentimiento_final': 'Sentimiento Final', 'count': 'Tweets'},
    barmode='group',
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_layout(template='plotly_white', xaxis_tickangle=-45, height=500)
fig.show()

In [ ]:
# 5. Heatmap: promedio de probabilidad negativa por candidato
import plotly.express as px

prob_por_candidato = df.groupby('candidato').agg(
    prob_positivo_media=('prob_positivo', 'mean'),
    prob_negativo_media=('prob_negativo', 'mean'),
    prob_neutro_media=('prob_neutro', 'mean'),
    confianza_media=('confianza_sentimiento', 'mean'),
    total_tweets=('tweet', 'count')
).round(3)

print("📊 Resumen de probabilidades promedio por candidato:")
print("=" * 70)
print(prob_por_candidato.to_string())

---
## Paso 7: Exportar el dataset enriquecido

In [ ]:
# Seleccionar columnas finales
df_export = df[['candidato', 'tweet',
                'sentimiento', 'prob_positivo', 'prob_negativo', 'prob_neutro',
                'confianza_sentimiento', 'emocion', 'prob_emocion_dominante',
                'sentimiento_final', 'fuente_prediccion']].copy()

df_export.to_csv('tweets_candidatos_con_features.csv', index=False)

print(f"✅ Dataset exportado: tweets_candidatos_con_features.csv")
print(f"   Filas: {len(df_export)}")
print(f"   Columnas: {len(df_export.columns)}")
print(f"\nColumnas del dataset final:")
for col in df_export.columns:
    print(f"   • {col}")

In [ ]:
df_export.head(10)

---
## Resumen y reflexiones finales

### ¿Qué aprendimos?

**1. Feature Engineering con modelos de NLP en español:** Transformamos una columna de texto libre (tweets) en múltiples features numéricas y categóricas usando `pysentimiento`, una librería diseñada para texto en español de redes sociales.

**2. Estrategia de modelos complementarios (de nuevo):** Igual que en el notebook de inferencia de género, usamos un modelo base (sentimiento) para clasificar todo y un modelo especializado (emociones) solo para profundizar en los casos interesantes. Esta es una estrategia reproducible:

```
Modelo rápido/general → clasifica todo → identifica subconjunto interesante
                                                     ↓
                                          Modelo profundo → enriquece
```

**3. De texto crudo a features analíticas:** Partimos de texto libre y generamos:
- 1 feature categórica base (`sentimiento`)
- 3 features numéricas de probabilidad (`prob_positivo`, `prob_negativo`, `prob_neutro`)
- 1 feature de confianza (`confianza_sentimiento`)
- 1 feature de emoción para negativos (`emocion`)
- 1 feature enriquecida combinada (`sentimiento_final`)

### Tabla resumen de los modelos

| Aspecto | Modelo 1 (Sentimiento) | Modelo 2 (Emociones) |
|---|---|---|
| Tarea | `sentiment` | `emotion` |
| Modelo base | RoBERTuito | RoBERTuito |
| Categorías | POS, NEG, NEU | joy, sadness, anger, fear, disgust, surprise, others |
| Tweets procesados | Todos (50) | Solo los negativos |
| Probabilidades | Sí (3 clases) | Sí (7 clases) |
| Uso | Clasificación general | Profundización de negativos |

### Limitaciones

- **Ironía y sarcasmo:** Los modelos de sentimiento tienen dificultades con el sarcasmo, que es muy frecuente en Twitter colombiano ("sí claro, este nos va a salvar").
- **Contexto:** El modelo analiza cada tweet de forma aislada. No sabe si el usuario es un militante o un opositor habitual.
- **Datos simulados:** En un proyecto real, tendríamos miles de tweets y podríamos calcular tendencias temporales.
- **Sesgo del modelo:** RoBERTuito fue entrenado mayoritariamente con tweets en español rioplatense (argentino). El español colombiano tiene matices distintos.

### Aplicaciones reales en ciencia política

- **Monitoreo de campaña:** Dashboard en tiempo real del sentimiento hacia cada candidato.
- **Detección de crisis:** Alertas cuando la proporción de rabia (`neg_anger`) supera un umbral.
- **Análisis de narrativas:** ¿Qué temas generan más miedo vs. más rabia?
- **Predicción electoral:** Las probabilidades de sentimiento como features en modelos de intención de voto.
- **Análisis de polarización:** ¿Qué candidatos dividen más la opinión pública?

### Para seguir explorando

- Prueba con `task="hate_speech"` para detectar discurso de odio en los tweets.
- Prueba con `task="irony"` para identificar tweets irónicos y filtrarlos del análisis.
- ¿Qué pasa si aplicas emociones también a los tweets positivos? ¿Hay diferencia entre "alegría" y "sorpresa positiva"?
- Intenta conectar este análisis con datos reales de Twitter/X usando su API o herramientas de scraping.